# 2.3 The article feature table

The price model consumes one row per article rather than one per sentence, and in this notebook we
make every decision about what goes in that row: which aggregations, over which populations, and how
many columns survive.

Three of the four decisions in section 1 cut features rather than adding them, and section 2 is a
design argument that we built on and then found describes 0.3% of the corpus.

We ran these cells when `AGGREGATED_VARIANTS` still promoted two variants, so some of the column
counts in the outputs predate the cut described in section 8. The reasoning and the ratios are
unaffected.

In [ ]:
import numpy as np
import pandas as pd

from stock_predictor.config import DATA_DIR, PROCESSED_DATA_DIR
from stock_predictor.text import fusion

pd.set_option("display.max_colwidth", 200)

sentences = pd.read_parquet(DATA_DIR / "sentences.parquet")
articles_meta = pd.read_parquet(
    PROCESSED_DATA_DIR / "processed_articles.parquet",
    columns=["article_id", "headline", "source"],
)
print("sentences:", sentences.shape, " articles:", articles_meta.shape)

### 1. Four decisions

**One variant rather than eight.** If we shipped all eight we would bloat the feature table with
constructions we have already shown to be dominated, and would invite overfitting the moment
anything downstream does feature selection. The first cut kept two, `conf_graft` and
`conf_graft_soft`, which are the two ends of one honest trade-off, and dropped `mean_blend` despite
its respectable showing, because it is a different idea rather than a point on the same trade-off.
The second cut took it to one, once `conf_graft_floor` was chosen in 2.2 section 12, because the
three conf-graft variants are the same formula at different floors and correlate at around 0.9, so
promoting all three spent twelve feature columns restating one number three times. They stay in
`VARIANTS` and are still returned per sentence, so every earlier measurement remains reproducible.

**No `excl_comp` twin.** The FinBERT family carried `sent_entity_excl_comp_*`, which is the target
mean recomputed with comparative sentences deleted. It existed because FinBERT reads "Build-A-Bear
outperformed Tesla" as positive news about Tesla, and with no way to tell whose sentiment it is,
throwing the sentence away was the only available defence. `conf_graft` takes ABSA's direction, and
ABSA gets 90% of the comparative disagreements right, so building the twin would delete precisely
the sentences the fusion handles well. The `excl_comp` family was later deleted outright along with
the comparative flag it was defined against.

**No `maxmag` twin.** `sent_entity_maxmag_*` stores the single target sentence with the largest
`|pos - neg|`. The instinct is right, since an article's loudest claim may matter more than its
average, but resting a feature on one sentence is noisy, and `conf_graft`'s compressed dynamic
range makes the single most extreme value a weaker signal for it than it is for FinBERT. Section 3
is what we put in its place.

**Keep `lead`.** It averages the first `LEAD_SENTENCE_WINDOW = 5` sentences and asks whether
position in the article matters, which is orthogonal to who did the scoring and nearly free to
compute.

We run every aggregation over the population `aggregate_article_features` uses for its
`sent_entity_*` family, which is `mentions_target & ~is_boilerplate`, restricted to rows where the variant's own
score is non-null. Sharing the population is deliberate, because it is what makes
`fus_conf_graft_floor_mean` directly comparable to `sent_entity_pos` and `sent_entity_neg` on the
same article: a difference between them is a difference in scoring method rather than a difference
in which sentences were read. Columns whose population is empty are NaN rather than 0, and counts
are the exception, since an empty count is a real zero.

In [2]:
# Per-sentence scores, kept separately so we can read individual sentences below.
# aggregate_fusion_features() calls score_variants() itself, so it takes the raw
# table -- handing it a frame that already carries the variant columns would
# collide on those names.
scored = fusion.score_variants(sentences)
sent = pd.concat([sentences.reset_index(drop=True), scored.reset_index(drop=True)], axis=1)

pop = sent[sent["mentions_target"] & ~sent["is_boilerplate"] & sent["conf_graft"].notna()]
print(f"population: {len(pop)} target sentences across {pop['article_id'].nunique()} articles")

feats = fusion.aggregate_fusion_features(sentences)
print("fusion feature columns:", [c for c in feats.columns if c != "article_id"])
feats.head()

population: 13947 target sentences across 1946 articles


fusion feature columns: ['fus_conf_graft_mean', 'fus_conf_graft_median', 'fus_conf_graft_lead', 'fus_conf_graft_top3_pos', 'fus_conf_graft_top3_neg', 'fus_conf_graft_spread', 'fus_conf_graft_soft_mean', 'fus_conf_graft_soft_median', 'fus_conf_graft_soft_lead', 'fus_conf_graft_soft_top3_pos', 'fus_conf_graft_soft_top3_neg', 'fus_conf_graft_soft_spread']


,article_id,fus_conf_graft_mean,fus_conf_graft_median,fus_conf_graft_lead,fus_conf_graft_top3_pos,fus_conf_graft_top3_neg,fus_conf_graft_spread,fus_conf_graft_soft_mean,fus_conf_graft_soft_median,fus_conf_graft_soft_lead,fus_conf_graft_soft_top3_pos,fus_conf_graft_soft_top3_neg,fus_conf_graft_soft_spread
0,138802119,-0.028223,0.000229,NaN,-0.028223,-0.028223,0.000000e+00,-0.020899,0.014257,NaN,-0.020899,-0.020899,0.000000
1,139021751,0.873508,0.873508,NaN,0.873508,0.873508,0.000000e+00,0.884834,0.884834,NaN,0.884834,0.884834,0.000000
2,138690782,-0.248157,-0.099860,NaN,-0.031625,-0.365824,3.341991e-01,-0.158841,-0.114440,NaN,0.096422,-0.384504,0.480925
3,136581238,-0.290023,-0.355874,-0.290023,-0.290023,-0.290023,-5.551115e-17,-0.366205,-0.449496,-0.366205,-0.366205,-0.366205,0.000000
4,136581240,-0.415505,-0.333902,-0.637844,-0.000075,-0.909551,9.094756e-01,-0.533573,-0.639167,-0.792887,0.001928,-0.930799,0.932727


### 2. Does the mean dilute?

A mean measures central tendency, but the question we actually want answered about an article is
closer to whether it carries a strong message, which is a property of the tail. Those come apart
badly, so consider two articles. Article A is short, six sentences, all mildly negative. Article B has
twenty sentences, three of them strongly negative and seventeen of neutral filler.

A reader would say B carries the stronger negative message, since it contains real bad news and the
filler is filler. The mean ranks A as more negative, because B's three strong sentences are divided
by twenty, so the signal is diluted by exactly the material a reader discounts.

So alongside the mean we take `_top3_neg` and `_top3_pos`, the average of the 3 most extreme
sentences in each direction. Three rather than one, because averaging three cuts the noise of
trusting a single sentence while still reading the tail. We keep both directions separately rather
than collapsing to an absolute value, because an article can carry strong positives and strong
negatives at once, and that mixed case is real information that `abs()` would destroy.

In [3]:
# Where do the two disagree most? Rank articles by each and compare.
cmp = feats.dropna(subset=["fus_conf_graft_mean", "fus_conf_graft_top3_neg"]).copy()
n_sents = pop.groupby("article_id").size().rename("n_target_sents")
cmp = cmp.merge(n_sents, on="article_id", how="left")

cmp["rank_mean"] = cmp["fus_conf_graft_mean"].rank()
cmp["rank_top3neg"] = cmp["fus_conf_graft_top3_neg"].rank()
cmp["rank_gap"] = cmp["rank_mean"] - cmp["rank_top3neg"]

print(f"articles compared: {len(cmp)}")
print(f"correlation of the two rankings: "
      f"{cmp['fus_conf_graft_mean'].corr(cmp['fus_conf_graft_top3_neg'], method='spearman'):.3f}")
print()
print("articles the MEAN ranks far more benign than the TAIL does (top 5):")
worst = cmp.nlargest(5, "rank_gap")[
    ["article_id", "n_target_sents", "fus_conf_graft_mean", "fus_conf_graft_top3_neg", "rank_gap"]
]
worst.round(3)

articles compared: 1946


correlation of the two rankings: 0.748

articles the MEAN ranks far more benign than the TAIL does (top 5):


,article_id,n_target_sents,fus_conf_graft_mean,fus_conf_graft_top3_neg,rank_gap
183,136913563,60,0.128,-0.826,1558.0
834,138291030,27,0.089,-0.901,1526.0
1549,140733574,22,0.117,-0.737,1462.0
1159,140124325,12,0.105,-0.647,1370.0
333,137258403,16,0.067,-0.741,1310.0


We look first at the article with the largest rank gap, and at first glance it is a textbook case: 60 target sentences, a mean of +0.128, a `top3_neg` of -0.826, and three substantive
negative sentences about declining deliveries.

In [4]:
# Read the most extreme case: what is the mean hiding?
case = cmp.nlargest(1, "rank_gap").iloc[0]
aid = case["article_id"]
print(f"article {aid}: {int(case['n_target_sents'])} target sentences, "
      f"mean {case['fus_conf_graft_mean']:+.3f}, top3_neg {case['fus_conf_graft_top3_neg']:+.3f}")
print()
rows = pop[pop["article_id"] == aid].sort_values("conf_graft")
print("--- its 3 most negative target sentences ---")
for _, r in rows.head(3).iterrows():
    print(f"  {r['conf_graft']:+.3f}  {r['text'][:150]}")
print()
print(f"--- and the {len(rows) - 3} others, which the mean averages in ---")
for _, r in rows.iloc[3:8].iterrows():
    print(f"  {r['conf_graft']:+.3f}  {r['text'][:150]}")

article 136913563.0: 60 target sentences, mean +0.128, top3_neg -0.826

--- its 3 most negative target sentences ---
  -0.893  Unfortunately for Tesla bulls, deliveries are 4% lower over the trailing twelve months, and over the past two years, growth has been sluggish at +1%.
  -0.824  Tesla deliveries peaked in Q4 2023 and have been declining ever since.
  -0.761  Oops, something went wrong A mere handful of months ago, Tesla ( TSLA ) shares looked doomed, and it felt as if the bears may have finally gotten thei

--- and the 57 others, which the mean averages in ---
  -0.580  The importance of Musk's presence at Tesla was put on full display when he stepped away from Tesla to focus on his acquisition of social media platfor
  -0.197  For value-oriented investors, Tesla's price-to-earnings ratio of 207x makes the stock an automatic avoid.
  -0.085  Many democrats felt that Musk wielded too much power, leading to mass left-wing protests against Tesla, boycotts, and widespread vandalism.

In [5]:
aid = int(cmp.nlargest(1, "rank_gap").iloc[0]["article_id"])
row = articles_meta[articles_meta["article_id"] == aid]
print("HEADLINE:", row["headline"].iloc[0])
print("SOURCE  :", row["source"].iloc[0])
print()

a = pop[pop["article_id"] == aid]
n_pos, n_neg = int((a["conf_graft"] > 0).sum()), int((a["conf_graft"] < 0).sum())
print(f"{len(a)} target sentences: {n_pos} positive, {n_neg} negative")
print()
print("--- its most POSITIVE sentences ---")
for _, r in a.nlargest(4, "conf_graft").iterrows():
    print(f"  {r['conf_graft']:+.3f}  {r['text'][:125]}")

HEADLINE: Tesla Will Crush Q3 Delivery Expectations: Here's Why
SOURCE  : Zacks

60 target sentences: 50 positive, 10 negative

--- its most POSITIVE sentences ---
  +0.733  In addition, TSLA shares are rising on optimism surrounding its newest 'Full Self Driving' (FSD) update, which is slated to l
  +0.730  Bottom Line After navigating a challenging period marked by political backlash, slowing EV growth, and market skepticism, Tes
  +0.708  Tesla shares up more than 30% in September and registered fresh all-time closing highs last week.
  +0.697  Since its inception, TSLA have increased by a mind-blowing 34k%.


Then we read the headline. It is a bull piece. Fifty positive sentences against ten, and the
positives are substantive: shares up 30% in a month, fresh all-time highs, an FSD update, thirteen
consecutive profitable quarters in Tesla Energy. The three damning sentences are the bear case the
article raises in order to argue against it.

So the mean of +0.128 is correct, and the article really is mildly positive. What `top3_neg` found
was not a diluted true message, it was the counterargument a persuasive article has to acknowledge.
A real dilution case needs the mean to say one thing while the article's strong content says
another.

In [6]:
# A TRUE dilution case: the mean is positive, but the article contains more
# loud-negative sentences than loud-positive ones (or vice versa).
LOUD = 0.4
g = pop.groupby("article_id")["conf_graft"]
stats = pd.DataFrame({
    "n": g.size(),
    "mean": g.mean(),
    "n_loud_neg": pop[pop["conf_graft"] < -LOUD].groupby("article_id").size(),
    "n_loud_pos": pop[pop["conf_graft"] > LOUD].groupby("article_id").size(),
}).fillna({"n_loud_neg": 0, "n_loud_pos": 0})
stats = stats[stats["n"] >= 8]

pos_mean_neg_tail = stats[(stats["mean"] > 0.02) & (stats["n_loud_neg"] > stats["n_loud_pos"])]
neg_mean_pos_tail = stats[(stats["mean"] < -0.02) & (stats["n_loud_pos"] > stats["n_loud_neg"])]

print(f"articles with >= 8 target sentences: {len(stats)}")
print(f"  mean POSITIVE but loud content net negative: {len(pos_mean_neg_tail)}")
print(f"  mean NEGATIVE but loud content net positive: {len(neg_mean_pos_tail)}")
n_conflict = len(pos_mean_neg_tail) + len(neg_mean_pos_tail)
print(f"  total conflicts: {n_conflict} ({100 * n_conflict / len(stats):.1f}% of articles)")

articles with >= 8 target sentences: 703
  mean POSITIVE but loud content net negative: 2
  mean NEGATIVE but loud content net positive: 0
  total conflicts: 2 (0.3% of articles)


We find two articles out of 703, which is 0.3%, and both of them are marginal.

The reason is worth stating because it is not obvious in advance: news articles are internally
coherent. A bearish article is bearish nearly all the way through, and a bullish one stays bullish
while conceding a few points. The pathological article, with three damning sentences hidden in
twenty of filler that says the opposite, is a shape financial journalism largely does not produce.
Boilerplate filler exists in abundance, but `flag_boilerplate` had already removed it before any of
this ran.

We keep the features, but we stop claiming they fix the mean. They are cheap, they are genuinely
distinct, since the two rankings correlate at 0.748 rather than 1.0, and they answer a question the
mean cannot: does this article contain strong statements about the target at all. That is not the
same as asking whether the mean is wrong, and it is the only version of the claim that survives.

### 3. The extremes matter as a pair

In section 2 we tested whether the tail contradicts the mean, and it barely ever does, which was
the wrong question to have asked.

Go back to the bull article we read above. Its `top3_neg` was -0.826, but its `top3_pos` was +0.723, so both ends
are loud. That is not a diluted article, it is a contested one, an article making strong claims in
both directions.

And here is the thing the mean genuinely cannot do. A contested article and a bland one both average
near zero. One is an earnings piece full of sharp claims that happen to offset, and the other barely
mentions the company, and the mean reports the same number for both. So the right statistic is not
either end on its own but the spread between them.

In [7]:
LOUD_N = 10
g2 = pop.groupby("article_id")["conf_graft"]
art_stats = pd.DataFrame({
    "n": g2.size(),
    "mean": g2.mean(),
    "median": g2.median(),
    "top3_pos": g2.apply(lambda x: x.nlargest(3).mean()),
    "top3_neg": g2.apply(lambda x: x.nsmallest(3).mean()),
})
art_stats["spread"] = art_stats["top3_pos"] - art_stats["top3_neg"]
art_stats["balance"] = art_stats["top3_pos"] + art_stats["top3_neg"]

big = art_stats[art_stats["n"] >= LOUD_N]
band = big[big["mean"].abs() < 0.05]
print(f"articles with >= {LOUD_N} target sentences: {len(big)}")
print(f"  of those, |mean| < 0.05 -- 'the mean says nothing': {len(band)}")
print(f"  their spread ranges {band['spread'].min():.2f} to {band['spread'].max():.2f}")
print()
contested = band[band["spread"] > band["spread"].quantile(0.75)]
quiet = band[band["spread"] < band["spread"].quantile(0.25)]
print(f"  {len(contested)} CONTESTED (loud both ways) vs {len(quiet)} QUIET (nothing strong)")
print("  -- indistinguishable by the mean, obviously different articles:")
for label, sub in [("CONTESTED", contested), ("QUIET", quiet)]:
    aid = sub.nlargest(1, "n").index[0]
    r = sub.loc[aid]
    h = articles_meta[articles_meta["article_id"] == aid]["headline"]
    print(f"\n  {label}: n={int(r['n'])} mean={r['mean']:+.3f} "
          f"top3_pos={r['top3_pos']:+.3f} top3_neg={r['top3_neg']:+.3f} spread={r['spread']:.3f}")
    print(f"    {h.iloc[0][:100] if len(h) else '?'}")

articles with >= 10 target sentences: 554
  of those, |mean| < 0.05 -- 'the mean says nothing': 213
  their spread ranges 0.04 to 1.71

  53 CONTESTED (loud both ways) vs 53 QUIET (nothing strong)
  -- indistinguishable by the mean, obviously different articles:

  CONTESTED: n=52 mean=+0.034 top3_pos=+0.753 top3_neg=-0.332 spread=1.085
    Earnings Season Surprises

  QUIET: n=25 mean=+0.020 top3_pos=+0.123 top3_neg=-0.000 spread=0.123
    Lightship's electric RVs are bringing campgrounds into the future


In [8]:
print("does spread carry information the mean lacks?")
print(f"  corr(|mean|, spread)  = {big['mean'].abs().corr(big['spread']):+.3f}   <- mostly new")
print(f"  corr(mean, balance)   = {big['mean'].corr(big['balance']):+.3f}   <- near-redundant")
print()
print("-> keep the DIFFERENCE of the extremes (spread); drop the SUM (balance).")

does spread carry information the mean lacks?
  corr(|mean|, spread)  = +0.319   <- mostly new
  corr(mean, balance)   = +0.937   <- near-redundant

-> keep the DIFFERENCE of the extremes (spread); drop the SUM (balance).


We get a correlation of 0.319 between `|mean|` and spread, so spread is mostly new information. The sum of
the extremes correlates 0.937 with the mean and is therefore near-redundant, so we keep `spread` as
an explicit column and drop `balance`. Explicit rather than left derivable, because tree-based
models cannot easily form a difference between two features on their own.

### 4. Why the top-K stays at 3

Top-5 is the obvious alternative, and the corpus argues against it.

In [9]:
n_sents = pop.groupby("article_id").size()
print("target sentences per article:")
print(n_sents.describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
print()
print("overlap: with top-K each way, the two ends share sentences when n < 2K")
for k in (3, 5):
    share = (n_sents < 2 * k).mean()
    print(f"  top-{k}: {int((n_sents < 2*k).sum())} articles ({100*share:.1f}%) have < {2*k} sentences")
print()
for c, f in [("top3_pos", lambda x: x.nlargest(3).mean()), ("top5_pos", lambda x: x.nlargest(5).mean()),
             ("top3_neg", lambda x: x.nsmallest(3).mean()), ("top5_neg", lambda x: x.nsmallest(5).mean())]:
    art_stats[c] = g2.apply(f)
b2 = art_stats[art_stats["n"] >= LOUD_N]
print(f"  corr(top3_pos, top5_pos) = {b2['top3_pos'].corr(b2['top5_pos']):.4f}")
print(f"  corr(top3_neg, top5_neg) = {b2['top3_neg'].corr(b2['top5_neg']):.4f}")

target sentences per article:
count    1946.0
mean        7.2
std         8.7
min         1.0
25%         1.0
50%         4.0
75%        10.0
90%        17.0
max       214.0

overlap: with top-K each way, the two ends share sentences when n < 2K
  top-3: 1071 articles (55.0%) have < 6 sentences
  top-5: 1392 articles (71.5%) have < 10 sentences



  corr(top3_pos, top5_pos) = 0.9797
  corr(top3_neg, top5_neg) = 0.9738


The median article has only 4 target sentences. If we used K=5 on a 4-sentence article both ends
would include every sentence and both collapse to the mean, so the feature destroys itself on 71.5% of the corpus,
and it buys nothing, since top-3 and top-5 correlate at 0.974 to 0.980. K=3 degrades gracefully by
comparison: at n of 3 or fewer both ends equal the mean exactly, which is a sensible fallback rather
than a wrong answer.

### 5. The median

In section 2 we went looking for the mean being distorted by extremes, by counting sign conflicts
between the mean and the loud content, and found 0.3%. There is a far more direct instrument. Compare the
mean to the median, and if a few extreme sentences are dragging the average, the two diverge.

In [10]:
med = art_stats[art_stats["n"] >= 5].copy()
sign_conflict = (np.sign(med["mean"]) != np.sign(med["median"])) & (med["median"] != 0)
print(f"articles with >= 5 target sentences: {len(med)}")
print(f"  corr(mean, median): {med['mean'].corr(med['median']):.3f}")
print(f"  mean and median DISAGREE ON SIGN: {int(sign_conflict.sum())} "
      f"({100*sign_conflict.mean():.1f}%)")
print()
print("|mean - median| divergence:")
print((med["mean"] - med["median"]).abs().describe(percentiles=[.5, .9]).round(4).to_string())

articles with >= 5 target sentences: 958
  corr(mean, median): 0.792
  mean and median DISAGREE ON SIGN: 206 (21.5%)

|mean - median| divergence:
count    958.0000
mean       0.0709
std        0.0626
min        0.0000
50%        0.0562
90%        0.1556
max        0.3726


206 of 958 articles, which is 21.5%, have a mean and a median that disagree on sign. In a fifth of
the corpus a handful of extreme sentences drag the average across zero relative to the robust
centre. That is the original dilution intuition, properly measured, and section 2's 0.3% was not
evidence the concern was wrong, it was evidence that counting sign conflicts against loud content is
a blunt way to detect it. The correlation of mean with median is 0.792, which confirms the two are
meaningfully distinct rather than redundant.

That gives us six aggregations per variant:

- **`_mean`** reads all target sentences and gives the article's overall tone.
- **`_median`** reads the same set and gives the tone robust to a few extreme sentences.
- **`_lead`** reads the first 5 sentences and asks whether the opening says something different.
- **`_top3_neg`** reads the 3 most negative and asks how bad the worst thing it says is.
- **`_top3_pos`** reads the 3 most positive and asks how good the best thing it says is.
- **`_spread`** is `top3_pos - top3_neg` and asks whether the article is contested or quiet.

Read as a pair, `_mean` and `_median` are themselves a skew detector. When they diverge, the
article's average is being set by its extremes rather than by its bulk.

### 6. The CEO population

Person-tier aliases never set `mentions_target`, as described in 2.0 section 3, but a sentence about
Musk in a Tesla article carries real signal, so the question is whether folding those sentences into
the target population would change anything.

The original ablation said 0.019 and 0.020, barely noticeable, and that was wrong. It used
`baseline.join(with_ceo, how="outer")` and then `.abs().mean()` on the difference. Articles with zero
target sentences of their own, which gain some only once CEO-only sentences are folded in, get an
outer-join NaN for the baseline, so their difference is NaN, and pandas' `.mean()` defaults to
`skipna=True`. Those rows vanished silently from the average that claimed to summarise them, and
they are exactly the articles where the choice matters most.

In [17]:
alt = scored_sentences.copy()
alt["mentions_target_incl_ceo"] = alt["mentions_target"] | alt["mentions_ceo"]

def agg_variant(df, target_col):
    g = df[df[target_col]].groupby("article_id")
    return g[["pos", "neg", "neu"]].mean()

baseline = agg_variant(alt, "mentions_target")
with_ceo = agg_variant(alt, "mentions_target_incl_ceo")

compare = baseline.join(with_ceo, lsuffix="_baseline", rsuffix="_with_ceo", how="outer")

touched_ids = set(alt.loc[alt.mentions_ceo & ~alt.mentions_target, "article_id"].unique())
compare_touched = compare.loc[compare.index.isin(touched_ids)]

both_defined = compare_touched.dropna(subset=["pos_baseline", "pos_with_ceo"])
ceo_only = compare_touched[compare_touched["pos_baseline"].isna() & compare_touched["pos_with_ceo"].notna()]

print(f"articles affected (gained target sentences from CEO-only mentions): {len(touched_ids)}")
print(f"  of which, already had SOME baseline target sentences: {len(both_defined)}")
print(f"  of which, had NO baseline target sentences at all (old bug's blind spot): {len(ceo_only)}")
print()
print("(a) mean absolute change, over articles where BOTH baseline and with_ceo are defined:")
print("  sent_entity_pos:", (both_defined["pos_with_ceo"] - both_defined["pos_baseline"]).abs().mean())
print("  sent_entity_neg:", (both_defined["neg_with_ceo"] - both_defined["neg_baseline"]).abs().mean())
print()
print(f"(b) CEO-only articles (n={len(ceo_only)}) -- baseline undefined, with_ceo net sentiment distribution:")
ceo_only_net = ceo_only["pos_with_ceo"] - ceo_only["neg_with_ceo"]
ceo_only_net.describe()


articles affected (gained target sentences from CEO-only mentions): 692
  of which, already had SOME baseline target sentences: 687
  of which, had NO baseline target sentences at all (old bug's blind spot): 5

(a) mean absolute change, over articles where BOTH baseline and with_ceo are defined:
  sent_entity_pos: 0.06054701891932218
  sent_entity_neg: 0.06492305557588599

(b) CEO-only articles (n=5) -- baseline undefined, with_ceo net sentiment distribution:


count    5.000000
mean     0.128459
std      0.143607
min     -0.012788
25%      0.039185
50%      0.097157
75%      0.161095
max      0.357645
dtype: float64

Split explicitly, we get a mean absolute change over the 687 articles where both are defined of
0.0605 and 0.0649, which is roughly three times the originally reported figure. The 5 articles the old
metric erased are reported here as what they are: articles that under the current default get
`sent_entity_*` of NaN, meaning no signal, when in fact they carry real sentiment about Musk-linked
content.

`sent_ceo_*` and `fus_ceo_mean` expose that directly per article, so a downstream model can decide
how much CEO talk is worth instead of us deciding here. We compute one statistic rather than six,
because the CEO population is empty for roughly 70% of articles, so the tail statistics would be NaN
nearly everywhere and near-duplicates of the mean where they are not.

### 7. The provenance split

`aggregate_fusion_features` pools every target sentence into one number per article, which silently
asserts that a sentence naming the company outright and a sentence a coreference model guessed at
are equally good evidence. We measured in notebook 2.1 section 4 that they are not. `surface`, where the
company is named literally, is unaudited. `coref_span`, resolved with a mention span, runs at 90.0%
on n=100. `coref_nospan`, resolved with no span, runs at 68.8% on n=170.

The three channels partition the target population, since an unset `resolved_by_coref` unambiguously
means an explicit mention, so the counts sum and the shares are meaningful.

This does not make the noisy channel less noisy, which is the judge's job. It stops the noise being
invisible, and that payoff holds either way: if the judge works, provenance records how much it
cleaned up, and if it does not, provenance is the fallback mitigation. A downstream model given the
split can down-weight or drop the weak channel without the text pipeline being re-run.

We split only the mean rather than all six aggregations, because splitting all six would give us 18
columns of unvalidated feature, and this module's own precedent is to keep the table narrow until
something earns its width. Three columns per channel over one promoted variant gives 12.

In [ ]:
# The provenance split over the real corpus.
from stock_predictor.text.fusion import (  # noqa: E402
    PROVENANCE_CHANNELS,
    aggregate_fusion_features,
    aggregate_provenance_features,
    provenance_channel,
    score_variants,
)

sent = sentences.copy()
sent["__channel"] = provenance_channel(sent)
sent["__cg"] = score_variants(sent)["conf_graft"]

pop = sent[sent["mentions_target"].fillna(False) & ~sent["is_boilerplate"].fillna(False)]
pop = pop.dropna(subset=["__cg"])

print(f"Non-boilerplate target sentences with a fusion score: {len(pop):,}\n")
print(f"{'channel':16s}{'n':>8s}{'share':>9s}{'mean conf_graft':>18s}")
for channel in PROVENANCE_CHANNELS:
    grp = pop[pop["__channel"] == channel]
    mean = f"{grp['__cg'].mean():+.4f}" if len(grp) else "--"
    print(f"{channel:16s}{len(grp):8,d}{len(grp)/len(pop):9.1%}{mean:>18s}")

# What does the noisy channel do to the article-level number it feeds?
by_article_all = pop.groupby("article_id")["__cg"].mean()
by_article_clean = pop[pop["__channel"] != "coref_nospan"].groupby("article_id")["__cg"].mean()
joined = pd.concat(
    [by_article_all.rename("with"), by_article_clean.rename("without")], axis=1
).dropna()
flips = np.sign(joined["with"]) != np.sign(joined["without"])
affected = pop[pop["__channel"] == "coref_nospan"]["article_id"].nunique()

print(f"\nDropping coref_nospan from the article-level blend:")
print(f"  articles compared                   : {len(joined):,}")
print(f"  articles containing any nospan row  : {affected:,}")
print(f"  correlation with / without          : {joined['with'].corr(joined['without']):.4f}")
print(f"  articles whose sentiment SIGN flips : {int(flips.sum())} "
      f"({flips.mean():.1%} of all, {int(flips.sum())/affected:.1%} of affected)")

prov = aggregate_provenance_features(sentences)
print(f"\naggregate_provenance_features: {len(prov):,} articles x {len(prov.columns)-1} columns")

The split is purely additive. `aggregate_fusion_features` returns byte-identical output with and
without the provenance columns present, and a test pins that.

It also does not move the headline. Dropping `coref_nospan` from the article-level blend leaves a
correlation of 0.9946 with 23 sign flips. What the split buys is separability rather than accuracy.

### 8. One score per population

The wide table has around 60 columns, and on 2,124 articles that leaves us about 33 rows per
feature.

Three redundancies made most of those columns unpayable for us. `pos`, `neg` and `neu` sum to 1, so one
column of every triple is determined by the other two. The raw triples and the fused score are three
readings of one sentence population, so feeding all three asks the model to rediscover a fusion we
already chose against 300 hand labels. And the promoted conf-graft variants correlate with each
other at around 0.9.

So the model reads one score per population, and every score is the same graft at `CONF_FLOOR`:

```
MODEL_FEATURE_COLUMNS = [
    n_total_sents, n_entity_sents, n_ceo_sents, n_boilerplate_sents,
    entity_share, article_length,
    fus_conf_graft_floor_{mean, median, lead, top3_pos, top3_neg, spread},
    fus_ceo_mean,
    fus_headline,
    fus_maxmag, fus_trusted_mean, fus_scorer_gap,
    fus_headline_gap, fus_lead_gap,
]
```

The last five were added after the fact, and they do not reopen the argument above: each is one
number over a population that already existed, not a new triple. `fus_maxmag` is the single loudest
target sentence, which is the tail `top3_pos` and `top3_neg` average over, read undiluted.
`fus_trusted_mean` is the mean over the surface and coref_span channels alone, which folds the
provenance split of section 7 into one column instead of nine. `fus_scorer_gap` is the mean absolute
distance between the two scorers, which is the one thing a fused score cannot express, since fusing
is exactly the operation that discards it. `fus_headline_gap` and `fus_lead_gap` are differences
against the body mean, explicit for the same reason `_spread` is explicit.

That takes the table to 19 features over 1,976 articles, or about 104 rows per feature, against the
33 the wide table would have given us.

The shape columns stay, because how much of an article is actually about the target is not sentiment
and cannot be recovered from a sentiment number.

The headline gets the same treatment as everything else. It is one string with no preceding context,
so there is no aggregation to do, one headline and one number. It had previously only ever gone
through FinBERT, which made it the one place in the pipeline where the sentence-level attribution
fix did not reach, despite carrying disproportionate weight per spec. `fus_headline` grafts both
scorers on the same string exactly as the sentence scorer does.

`build_model_features` composes all of this from the same functions the wide table uses, so the two
can never disagree.

### 9. What the full run writes

`stock_predictor/text/run_pipeline.py` turns every decision above into files. It reads the cleaned
article table at `config.PROCESSED_ARTICLES_PATH`, runs five phases in order, and writes the
intermediates into `data/interim/full_run/` and the deliverable into `data/processed/pipeline_run/`.

Each sentence file is the previous one plus the columns its phase adds, over the same 71,410 rows:

- **`sentences_tagged.parquet`**, phase A: the 10 `SENTENCE_COLUMNS`. `process_articles` splits every
  article, tags it, resolves coreference over the full body, and flags boilerplate. The only phase
  that reads article text.
- **`sentences_scored.parquet`**, phase B: 18 columns, adding FinBERT's `pos`, `neg`, `neu` and
  DeBERTa's five `absa_*`. Only rows passing `needs_score()` carry numbers, 16,660 of them, so 76.7%
  are skipped.
- **`sentences_judged.parquet`**, phase C: 21 columns, adding `provenance_channel`, `judge_answer`
  and `judge_accepted`. We ask the judge about the 3,553 coref rows defined in 2.1 and it accepts
  1,883, or 53.0%.
- **`articles_ungated.parquet`** and **`articles_judge_gated.parquet`**, phase D: 2,124 articles over
  47 columns each, the first aggregated over every sentence and the second over accepted rows alone.

The two article tables differ only in which sentences reached the aggregation, and neither drops an
article. A rejected sentence keeps its scores and loses only its contribution, so an article the gate
empties still appears with its features unset, which here is 179 of the 2,124.

Phase E writes what this notebook has been arguing towards:

- **`article_features.parquet`**: 1,976 articles over 23 columns, four identity and the 19 features
  of section 8. Built from accepted sentences alone, carrying no raw probability triples and no
  provenance columns, sorted by `timestamp_utc` so the market layer joins on it directly. The row
  count is below the corpus size because the relevance filter drops articles that neither mention the
  target in the body nor name it in the headline.
- **`article_features.md`**: its data dictionary, generated from the frame rather than written by
  hand, so a column and its description cannot drift apart.

This is the only output meant to leave the text layer. Everything in `full_run/` exists so this file
can be rebuilt or audited.

Four caches sit outside `full_run/` and are what make a second run cheap. Three are keyed on the hash
of what was scored, so re-running after a tagging change re-scores only the sentences whose text
moved. The fourth is the one that matters: the judge cache holds 3,553 verdicts keyed on
`article_id`, `sent_idx`, `target`, `model_id` and `prompt_version`, and a cold pass costs about
4 hours 50 minutes at 5.2 seconds per row against roughly 23 minutes for every other phase combined.
Verdicts flush every 100 rows, so an interrupted run resumes. Putting the model and prompt version in
the key is what stops verdicts formed under one prompt being served for another.

Two coverage numbers are worth knowing before modelling. `fus_ceo_mean` is populated for 49.4% of
rows, since most articles never mention the CEO apart from the target, and
`fus_conf_graft_floor_lead` is fully populated only because an empty lead window scores 0.0 rather
than NaN, so roughly a third of that column is a point mass at zero.

### 10. What this does not settle

Everything here is a design argument plus a demonstration that the features are distinct. None of it
is evidence that any of them predicts anything.

The choice among the six aggregations is entirely unresolved. Section 5 establishes that `_median`
and `_spread` carry information `_mean` does not, at 21.5% sign disagreement and a correlation of
0.319 respectively, but carrying different information is not predicting better. It is perfectly
possible that the plain mean is the most predictive of the six and the rest are elaborate noise.
To resolve it we need forward returns, which live in the market layer, and no price model exists
yet, so every quality claim in this series is a proxy we measured on hand labels rather than an
outcome.

Section 2 is also a caution about our own method. We built on the dilution argument, then found it
occurs in 0.3% of articles, then rescued it in a different form with the median. Two of the three
checks changed what we believed, so design arguments that sound compelling are worth checking
against the corpus before being trusted.

There is a structural seam running through the whole pipeline, and it bounds everything above. We
tag with full article context and then score without it. `process_articles` hands the entire article body
to the coreference model before any sentence splitting, so a sentence reading "The company also
raised prices" is correctly identified as a Tesla sentence on the strength of a mention three
sentences earlier. But FinBERT and ABSA then receive that sentence alone, with nothing around it.
The substitution described in 2.1 section 2 exists precisely to patch this, restoring the company's
identity to a scorer that cannot see the article, but not the surrounding narrative that gave the
sentence its meaning.

The consequence is a real ceiling. An isolated sentence like "It's a wonderful business" carries no
recoverable signal even when its tag is perfectly correct, because the information that made it
meaningful lives in the neighbouring sentences. No amount of improving the scorer fixes that, since
the input is impoverished before the model sees it.

The experiment we think is worth running is to score overlapping windows of 2 to 3 sentences rather
than single sentences, and aggregate the window scores back to the sentence or the article. We should know the cost before anyone starts. The cache is keyed on sentence text, so window scoring invalidates it
and forces a cold pass. Overlapping windows mean each sentence appears in several scores and needs a
defined aggregation. Per-sentence attribution blurs, which matters for the `top3` and `spread`
features built above. And window boundaries would need to respect article boundaries. We flag it
rather than doing it, because it is a change to the scoring substrate that would invalidate every
number in 2.2 and 2.3, and it should be attempted with a downstream metric in place to judge whether
it helped.

Two fixes found across the series remain unattempted. The first is a publisher-boilerplate blocklist,
described in 2.0 section 4, which would normalise numbers before counting exact text, and which is
also the principled fix for the off-target regression `CONF_FLOOR = 0.7` introduces. The second is a
negation and concession check, since ABSA reliably inverts on "excluding Tesla" and on "despite
selling a fraction of the cars". Together these plausibly take the measured harmful rate from 5.51%
to around 2 to 3%, which is more error removed than the entire four-stage coreference programme in
2.1 achieved.

### 11. What was removed along the way

Several mechanisms were built, measured and then cut. They are recorded here because the reasoning
is the useful part, and because two of them will be proposed again by anyone reading the code
without this history.

**Other-company detection, and the `excl_comp` families.** The sentence schema carried
`mentions_other`, `is_comparative`, `other_source` and `other_key`, and the article table carried
`sent_other_mean_*`, `absa_other_mean_*` and two exclusion families built on them. All of it existed
to stop a rival's good news, sitting in a Tesla article, scoring as Tesla's good news, which FinBERT
cannot avoid because it scores a whole sentence and cannot tell whose sentiment it reads. ABSA scores
toward an aspect and is handed the target explicitly, so it gets those sentences right by
construction, and the justification was superseded rather than outgrown. The measurements that
decided it: of 10,502 detector-sourced non-boilerplate rows only 1,369 also mentioned the target, so
9,133 existed purely to feed an unvalidated family; and removing the detector unblocked coreference
on 663 sentences it had been pre-empting, 344 of which resolve to the target, so it was costing real
signal rather than being merely inert.

**The spaCy ORG detector** went with it, at 50% to 65% precision. Two things were deliberately not
done, and both are live footguns:

- **spaCy's `ner` component stays enabled.** PERSON entities are load-bearing in three places, and
  disabling `ner` to tidy up after the ORG detector breaks all three silently. `_get_nlp()` says so.
- **`config.COMPANIES` was not emptied.** It survives as a registry of potential targets, which is
  what makes the pipeline ticker-agnostic. What went is the code path that read non-target entries
  as rivals.

**The gold set.** Removed because the design was wrong, not because it was redundant: the sampler
showed sentences without their surrounding context and then asked whether the attribution was
right, which is a judgement that often requires the preceding sentences. It was abandoned unlabelled
rather than labelled and trusted. The transferable lesson, and it cost three attempts to learn: when
an evaluation says almost everything is broken, suspect the evaluation first.

**The sourced company registry.** A module that built an alias table from a whole exchange listing,
with a screening pass deciding which bare names were safe to match. Measured net harmful, the
concrete failure being that it tagged Tesla's own Optimus robot as a rival, and deleted on
2026-08-21. The error tail in section 5 of 2.1 is independent evidence against curated rosters: 68
distinct wrong referents across 85 errors is not a list anyone can maintain.

**The anaphora recency heuristic.** A rule that attached an unresolved pronoun to the most recent
named company. Audited against coreference on equivalent samples it scored 13 of 100 against
coreference's 74 of 100, and the passage was determinate in 99% of cases, so that gap is not
labeller uncertainty. Worse than the miss rate: 61% of its failures resolved to a different named
company, so it was confidently wrong rather than merely absent. It was run dormant behind a flag for
a time and then removed entirely.

**Maverick coreference.** An alternative backend trialled at length. Two recommendations came out of
it, an ensemble-disagreement gate and a backend swap, and both were withdrawn by the same programme
once the judge measured better on the same rows. Nothing shipped, and `COREF_MODEL` is unchanged.
What is worth keeping is the shape of the decision: a pre-registered rule that separated "genuinely
better backend" from "merely more conservative backend" before the numbers arrived.

**Smaller cuts**, with the reason each will look tempting again:

| mechanism | fate | why it stays cut |
|---|---|---|
| `gated` fusion variant | live as a column, never promoted | Identical to `fin` in every stratum, and structurally cannot flip a sign. Someone will propose this again. |
| `sign_graft` deadband | superseded by confidence weighting | `DEADBAND = 0.0`. Directional accuracy falls monotonically as the deadband grows. |
| `mean_blend` carried alongside | never built | An earlier draft recommended carrying it; section 8 above drops it. |
| First-mention-wins antecedent rule | superseded | Replaced by subject position from the parse. |
| `score_headlines` cache truncation | fixed | Both save paths now merge rather than replace. |

Bringing any of these back needs new evidence, not a fresh argument. The measurements above are what
they were cut on.